# STAT 248 — Lock team–game panel & time index

**Goals (this notebook):**

1. **Observation unit:** one row = one team’s side of one regular‑season game.
2. **Sort key:** `(TEAM_ID, SEASON_ID, GAME_DATE, GAME_ID)` so each team‑season forms a chronological series.
3. **Sanity checks:** no duplicate `(GAME_ID, TEAM_ID)`, dates monotone within team‑season.
4. **Variables used later:** fatigue (`days_rest`, `is_back_to_back`, `is_short_rest`, opponent counterparts), outcomes (`point_diff`, `PLUS_MINUS` if present, `EFG_PCT`, `TOV`), context (`IS_HOME`, rolling features from `build_nba_team_game_dataset.py`).

Set the notebook working directory to the `nba_api` repo root (the folder containing `scripts/`).

In [ ]:
from pathlib import Path

import pandas as pd

import sys

REPO_ROOT = Path.cwd().resolve()
_build = REPO_ROOT / "scripts" / "build_nba_team_game_dataset.py"
_ps = REPO_ROOT / "scripts" / "panel_structure.py"
if not _build.exists() or not _ps.exists():
    raise RuntimeError(
        f"Working directory looks wrong:\n{_build=} exists={_build.exists()}, " 
        f"{_ps=} exists={_ps.exists()}\nCd to the repo root and restart."
    )

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from panel_structure import PANEL_SORT_KEYS, sort_panel, validate_team_game_panel

## Refresh data (NBA.com via `nba_api`)

`REBUILD_PANEL = True` will call the builder script once (rates API). Set `False` to only read CSV.

In [ ]:
import subprocess

SEASONS = ["2022-23", "2023-24", "2024-25"]
_out_dir = REPO_ROOT / "data"
_out_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = _out_dir / "nba_team_game_panel_stat248.csv"

REBUILD_PANEL = False

import os

if REBUILD_PANEL:
    cmd = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "build_nba_team_game_dataset.py"),
        "--seasons",
        *SEASONS,
        "--rolling-window",
        "5",
        "-o",
        str(OUTPUT_CSV),
        "--season-delay",
        "0.65",
        "--timeout",
        "90",
    ]
    env = {**os.environ, "PYTHONPATH": str(REPO_ROOT / "src")}
    subprocess.run(cmd, check=True, cwd=str(REPO_ROOT), env=env)

In [ ]:
if not OUTPUT_CSV.exists():
    raise FileNotFoundError(
        f"No CSV at {OUTPUT_CSV}. Flip REBUILD_PANEL to True once to download/build."
    )

df_raw = pd.read_csv(OUTPUT_CSV, parse_dates=["GAME_DATE"])

df = sort_panel(df_raw)
print("PANEL_SORT_KEYS:", PANEL_SORT_KEYS)

In [ ]:
summary = validate_team_game_panel(df, sample_series=True, series_sample_n=3)
pd.Series({k: v for k, v in summary.items() if k != "series_sample_heads"}, dtype=object)
summary["series_sample_heads"]

## Quick structure summary

In [ ]:
# `summary` is from validation cell above (counts + sample heads already shown).
assert len(df) == summary["n_rows"]

In [ ]:
# Team–season counts (distinct time series units)
_n_team_season = df.groupby(["TEAM_ID", "SEASON_ID"], sort=False).ngroups
print("n_team_season_series:", _n_team_season)
print("SEASON breaks:")
print(df["SEASON_ID"].value_counts().sort_index())

for col in [
    "days_rest",
    "is_back_to_back",
    "is_short_rest",
    "opp_days_rest",
    "opp_is_back_to_back",
    "opp_is_short_rest",
]:
    if col in df.columns:
        print()
        print(col, df[col].value_counts(dropna=False).sort_index().head(20))

### Time dependence note (why `days_rest` is NaN exactly once per team-season)
`days_rest = (calendar gap − 1 between games)`. The first scheduled game each season has no prior game → `NaN` by design (`is_*` fatigue flags exclude those rows).

In [ ]:
# Align with proposal outcomes (primary + mechanisms)
proposal_cols = [c for c in ("point_diff", "PLUS_MINUS", "EFG_PCT", "TOV", "FG_PCT") if c in df.columns]
pd.concat([df[proposal_cols].describe().T["count"]], axis=1, keys=["n_nonnull"])